# Link adaptation

> The rate of each transmission (e.g. its MCS): the fastest meeting a target error rate, one for all the copies of a broadcast; and the MCSs of 5G NR.

In [ ]:
#| default_exp link_adaptation

In [ ]:
#| hide
from nbdev.showdoc import *

A PHY abstraction with several rates (e.g. the MCSs of a table) rates each transmission at every one of them.
`LinkAdaptation` then picks one per transmission: the fastest whose error rate meets a target, or the slowest if
none does. The copies of a broadcast get one rate, the fastest meeting the target at every receiver: the weakest
receiver's. `SionnaErrorModel` uses it (see `phy_abstraction`); it works on numpy arrays and torch tensors.

It is the rule of Sionna SYS's `InnerLoopLinkAdaptation` (`sionna.sys.link_adaptation`), which also has an outer
loop correcting the SINR from acknowledgements (OLLA); one would go here.

In [ ]:
#| export
from __future__ import annotations

import math
from collections import defaultdict
from typing import Sequence

import numpy as np

from comm_core.block import Block, _is_torch

In [ ]:
#| export
class LinkAdaptation(Block):
    "The fastest rate meeting a target error rate (else the slowest): one per transmission, one for all the copies of a broadcast."

    outputs = ('choice',)

    def __init__(
        self,
        target: float = 0.1, # The highest error rate allowed
    ):
        self.target = target

    def call(
        self,
        error_rate, # Error rate of each transmission at each rate, (n, M), the rates from slowest to fastest (numpy or torch)
        group: Sequence | None = None, # The broadcast of each transmission, None for a unicast (None: all unicasts)
    ):
        "Index of the rate chosen for each transmission, (n,)."
        ok = error_rate <= self.target
        if group is not None:                           # a broadcast's copies: a rate meeting the target at all of them
            copies = defaultdict(list)
            for k, g in enumerate(group):
                if g is not None:
                    copies[g].append(k)
            for ks in copies.values():
                ok[ks] = ok[ks].all(0)
        if _is_torch(ok):
            import torch
            fastest = (ok * torch.arange(ok.shape[1], device=ok.device)).argmax(1)
            return torch.where(ok.any(1), fastest, torch.zeros_like(fastest))
        fastest = (ok * np.arange(ok.shape[1])).argmax(1)
        return np.where(ok.any(1), fastest, 0)

In [ ]:
show_doc(LinkAdaptation.call)

In [ ]:
rates = np.array([[0.0, 0.05, 0.5], [0.0, 0.2, 0.9], [0.5, 0.9, 1.0], [0.0, 0.0, 0.01]])
la = LinkAdaptation(target=0.1)
assert la(rates).tolist() == [1, 0, 0, 2]                                 # none meets it (row 2): the slowest
assert la(rates, group=['x', 'x', None, 'y']).tolist() == [0, 0, 0, 2]     # a broadcast: its weakest receiver's

## The MCSs of 5G NR

The rates of 5G NR are the MCSs (modulation and coding schemes) of its tables. `NRMCSTable` holds a table's MCSs,
their modulation orders `Q_m` and code rates `R_m` (Sionna's `MCSDecoderNR`), and the resource elements a payload
takes at each (`nr_allocation`): the fewest, one subcarrier for one OFDM symbol each, whose NR transport block holds
it at an effective code rate of at most 0.95, CRCs included.

Both levels use it, so they agree on what an MCS costs: `SionnaErrorModel` rates each MCS from its table at the
system level (see `phy_abstraction`), and `SionnaLink` codes the bits at the MCS chosen (see `link_level`). It needs
Sionna (PyTorch): `pip install 'comm-core[sionna]'`.

In [ ]:
#| export
MAX_CODERATE = 0.95                                        # NR: a UE may skip decoding above this effective code rate


def nr_allocation(
    num_bits: int, # Payload size
    modulation_order: int, # Bits per QAM symbol of the MCS
    coderate: float, # Code rate of the MCS
) -> tuple:
    "(resource elements, TB size): the fewest resource elements whose NR transport block holds `num_bits`."
    import torch
    from sionna.phy.nr.utils import calculate_tb_size
    num_bits = max(int(num_bits), 1)
    first = max(1, math.ceil(num_bits / (modulation_order * coderate)))
    for lo in range(first, first + 100_000, 2048):         # candidates in chunks; usually the first ones fit
        re = torch.arange(lo, lo + 2048)
        tb, _, num_cb, tb_crc, cb_crc = calculate_tb_size(
            modulation_order=torch.full_like(re, modulation_order), target_coderate=torch.full(re.shape, float(coderate)),
            num_coded_bits=re * modulation_order, return_cw_length=False, verbose=False)
        rate = (tb + tb_crc + num_cb * cb_crc) / (re * modulation_order)
        fits = (tb >= num_bits) & (rate <= MAX_CODERATE)
        if fits.any():
            i = int(torch.nonzero(fits)[0])
            return int(re[i]), int(tb[i])
    raise ValueError(f"no NR allocation for {num_bits} bits")


class NRMCSTable:
    "The MCSs of a 5G NR MCS table: their modulation orders and code rates, and the resource elements of a payload at each (cached)."

    def __init__(
        self,
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        mcs_indices: Sequence[int] | None = None, # The MCSs to keep; None: all of the table's
        min_coderate: float = 0.0, # The MCSs with lower code rates are left out (e.g. those an encoder can't do)
        device: str | None = None, # Torch device; None: Sionna's default
    ):
        import torch
        from sionna.phy.nr.utils import MCSDecoderNR
        decoder = MCSDecoderNR(device=device)
        self.mcs_table_index, self.mcs_category = mcs_table_index, mcs_category
        self.indices, self.modulation_order, self.coderate = [], [], []
        for m in range(32):                                # the MCS indices of this table
            try:
                q, r = decoder(torch.tensor([m], dtype=torch.int32, device=device), mcs_table_index=mcs_table_index,
                               mcs_category=mcs_category, check_validity=True)
            except ValueError:
                continue
            if (mcs_indices is None or m in set(mcs_indices)) and float(r) >= min_coderate:
                self.indices.append(m)
                self.modulation_order.append(int(q))
                self.coderate.append(float(r))
        self._allocations = {}                             # (bits, MCS) -> (resource elements, TB size)

    def __len__(self) -> int:
        return len(self.indices)

    def mcs(
        self,
        mcs: int, # MCS index
    ) -> tuple:
        "(modulation order, code rate) of an MCS."
        i = self.indices.index(mcs)
        return self.modulation_order[i], self.coderate[i]

    def allocation(
        self,
        num_bits: int, # Payload size
        mcs: int, # MCS index
    ) -> tuple:
        "(resource elements, TB size) of a payload at an MCS (see `nr_allocation`)."
        if (num_bits, mcs) not in self._allocations:
            self._allocations[num_bits, mcs] = nr_allocation(num_bits, *self.mcs(mcs))
        return self._allocations[num_bits, mcs]

In [ ]:
show_doc(nr_allocation)

In [ ]:
show_doc(NRMCSTable.allocation)

In [ ]:
#| sionna
table = NRMCSTable()                                             # table 1, PUSCH
assert table.indices[0] == 0 and table.mcs(0)[0] == 2 and len(table) == 29        # MCS 0-28
assert NRMCSTable(min_coderate=0.2).indices[0] == 3                # code rates below 1/5 left out
assert NRMCSTable(mcs_indices=[5, 10]).indices == [5, 10]
re, tb = table.allocation(1000, 10)
q, r = table.mcs(10)
assert tb >= 1000 and re >= 1000 / (q * r) and table.allocation(1000, 10) == (re, tb)     # cached

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()